# ICU Mortality Prediction Pipeline

This notebook demonstrates a realistic end-to-end use of `clinops` to build a dataset
suitable for predicting in-hospital mortality from the first 48 hours of an ICU stay.

The workflow follows standard clinical ML practice:

```
raw EHR data
  → cohort definition (ICU stays with ≥48h data)
  → ingest & validate
  → preprocess (clip outliers, normalize units, harmonize ICD codes)
  → align to ICU admission anchor
  → extract temporal feature windows
  → imputation (train-set fit only)
  → lag/rolling features
  → attach outcome label
  → patient-stratified train/test split
  → ML-ready arrays
```

**No MIMIC-IV access is required.** Synthetic data that mirrors MIMIC-IV structure is
generated inline. Swap in a real `MimicTableLoader` path to run against real data.

---

In [ ]:
# Install clinops if not already installed
# !pip install clinops

import pathlib
import tempfile
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

print("Imports OK")

## Step 1 — Synthetic MIMIC-IV-style data

We generate synthetic data for 20 ICU patients. Each patient has:
- Hourly vital-sign measurements over 60 hours in `chartevents`
- An ICU stay record in `icustays` with admission (`intime`) and discharge (`outtime`) times
- A hospital admission record in `admissions` with binary outcome (`hospital_expire_flag`)
- 1–3 diagnoses in `diagnoses_icd`, mixing ICD-9 and ICD-10 codes

Non-survivors (patients 1–4) have subtly worse vitals on average, reflecting real
physiological deterioration patterns.

In [ ]:
rng = np.random.default_rng(99)

N_PATIENTS   = 20
N_HOURS      = 60
BASE_TIME    = datetime(2150, 3, 1, 6, 0)   # MIMIC uses fictional future dates
NON_SURVIVOR = set(range(1, 5))             # patients 1–4 die in hospital

# ── chartevents ────────────────────────────────────────────────────────────
rows = []
for pid in range(1, N_PATIENTS + 1):
    deteriorating = pid in NON_SURVIVOR
    for h in range(N_HOURS):
        hr_mean  = 95 + h * 0.3 if deteriorating else 72
        spo2_mean = 93 - h * 0.05 if deteriorating else 97
        rows.append({
            "subject_id": pid,
            "hadm_id":    1000 + pid,
            "stay_id":    2000 + pid,
            "charttime":  BASE_TIME + timedelta(hours=h),
            "heart_rate": float(np.clip(rng.normal(hr_mean,  10), 40, 200)),
            "spo2":       float(np.clip(rng.normal(spo2_mean, 2), 80, 100)),
            "resp_rate":  float(np.clip(rng.normal(18 if deteriorating else 14, 3), 5, 60)),
            "map":        float(np.clip(rng.normal(72 if deteriorating else 88, 12), 30, 180)),
        })

chartevents = pd.DataFrame(rows)

# Inject some impossible values for the outlier clipper to handle
chartevents.loc[10, "heart_rate"] = 320.0
chartevents.loc[50, "spo2"]       = 15.0

# ── icustays ───────────────────────────────────────────────────────────────
icustays = pd.DataFrame({
    "subject_id":      range(1, N_PATIENTS + 1),
    "hadm_id":         range(1001, 1001 + N_PATIENTS),
    "stay_id":         range(2001, 2001 + N_PATIENTS),
    "first_careunit":  ["MICU"] * N_PATIENTS,
    "last_careunit":   ["MICU"] * N_PATIENTS,
    "intime":          [BASE_TIME] * N_PATIENTS,
    "outtime":         [BASE_TIME + timedelta(hours=N_HOURS)] * N_PATIENTS,
    "los":             rng.uniform(1.5, 10, N_PATIENTS).round(2),
})

# ── admissions ─────────────────────────────────────────────────────────────
admissions = pd.DataFrame({
    "subject_id":          range(1, N_PATIENTS + 1),
    "hadm_id":             range(1001, 1001 + N_PATIENTS),
    "admittime":           [BASE_TIME] * N_PATIENTS,
    "dischtime":           [BASE_TIME + timedelta(days=6)] * N_PATIENTS,
    "admission_type":      ["EMERGENCY"] * N_PATIENTS,
    "hospital_expire_flag": [1 if pid in NON_SURVIVOR else 0
                             for pid in range(1, N_PATIENTS + 1)],
})

# ── diagnoses_icd (mixed ICD-9 and ICD-10) ─────────────────────────────────
dx_pool = [
    ("I509", 10), ("4280", 9),   # Heart failure
    ("E119", 10), ("25000", 9),  # Type-2 diabetes
    ("N179", 10), ("5849", 9),   # Acute kidney injury
    ("J189", 10), ("486",  9),   # Pneumonia
]
dx_rows = []
for pid in range(1, N_PATIENTS + 1):
    chosen = rng.choice(len(dx_pool), size=rng.integers(1, 4), replace=False)
    for rank, idx in enumerate(chosen, start=1):
        code, ver = dx_pool[idx]
        dx_rows.append({"subject_id": pid, "hadm_id": 1000 + pid,
                         "seq_num": rank, "icd_code": code, "icd_version": ver})
diagnoses_icd = pd.DataFrame(dx_rows)

print(f"chartevents : {len(chartevents):,} rows  ({chartevents['subject_id'].nunique()} patients)")
print(f"icustays    : {len(icustays):,} rows")
print(f"admissions  : {len(admissions):,} rows  (non-survivors: {admissions['hospital_expire_flag'].sum()})")
print(f"diagnoses   : {len(diagnoses_icd):,} rows")

## Step 2 — Ingest and validate

In production you would load directly from MIMIC-IV:

```python
from clinops.ingest import MimicTableLoader

tbl    = MimicTableLoader("/data/mimic-iv-2.2")
charts = tbl.chartevents(subject_ids=cohort_ids)
stays  = tbl.icustays(subject_ids=cohort_ids, with_los_band=True)
adm    = tbl.admissions(subject_ids=cohort_ids)
dx     = tbl.diagnoses_icd(subject_ids=cohort_ids)
```

Here we use `FlatFileLoader` + `ClinicalSchema` to validate our synthetic data.
The schema enforces nullability and physiological range constraints before any
processing happens.

In [ ]:
import os
from clinops.ingest import ClinicalSchema, ColumnSpec, FlatFileLoader

vitals_schema = ClinicalSchema(
    name="chartevents_vitals",
    columns=[
        ColumnSpec("subject_id",  nullable=False),
        ColumnSpec("charttime",   nullable=False),
        ColumnSpec("heart_rate",  min_value=0,  max_value=300),
        ColumnSpec("spo2",        min_value=50, max_value=100),
        ColumnSpec("resp_rate",   min_value=0,  max_value=80),
        ColumnSpec("map",         min_value=20, max_value=250),
    ],
)

with tempfile.NamedTemporaryFile(suffix=".csv", mode="w", delete=False) as f:
    chartevents.to_csv(f.name, index=False)
    tmp_csv = f.name

# strict=False logs violations (would raise SchemaValidationError if strict=True)
loader = FlatFileLoader(tmp_csv, schema=vitals_schema, id_col="subject_id", strict=False)
charts = loader.load()
print(loader.summary())
os.unlink(tmp_csv)

## Step 3 — Preprocess

### 3a. Clip physiologically impossible values

We injected a heart rate of 320 and an SpO2 of 15 earlier. These would corrupt
any model trained on them. The clipper replaces them with the physiological upper
or lower bound.

In [ ]:
from clinops.preprocess import ClinicalOutlierClipper

clipper = ClinicalOutlierClipper(action="clip")
charts  = clipper.fit_transform(charts)

print("Outlier audit:")
print(clipper.report().to_string(index=False))

### 3b. Harmonize ICD codes

MIMIC-IV `diagnoses_icd` contains both ICD-9 (for admissions before 2015) and ICD-10
(for admissions from 2015 onwards). We need a single coding system for downstream
comorbidity features.

The chapter assignment (Circulatory, Endocrine, etc.) lets us build high-level
one-hot comorbidity flags without needing to enumerate every specific code.

In [ ]:
from clinops.preprocess import ICDMapper

mapper     = ICDMapper()
diagnoses  = mapper.harmonize(diagnoses_icd, code_col="icd_code", version_col="icd_version")
diagnoses["chapter"] = mapper.chapter_series(diagnoses["icd_code"])

print(f"Diagnoses after harmonization ({mapper.n_mappings:,} curated mappings):")
print(diagnoses.groupby("chapter")["subject_id"].count().rename("patient_count").reset_index().to_string(index=False))

### 3c. Build comorbidity features from ICD chapters

One-hot encode ICD chapters per patient — a simple but effective set of
comorbidity features that is invariant to ICD version.

In [ ]:
# Create a binary matrix: one row per patient, one column per ICD chapter
comorbidities = (
    diagnoses
    .groupby(["subject_id", "chapter"])
    .size()
    .gt(0)
    .astype(int)
    .unstack(fill_value=0)
    .reset_index()
)
# Rename columns for clarity
comorbidities.columns = (
    ["subject_id"] +
    [f"dx_{c.lower().replace(' ', '_').replace('/', '_')}" for c in comorbidities.columns[1:]]
)

print(f"Comorbidity matrix: {comorbidities.shape}")
comorbidities.head()

## Step 4 — Cohort alignment

We want to use only the **first 48 hours** of ICU measurements. `CohortAligner` trims
each patient's time-series to a window around their ICU admission time (`intime`) and
adds an `hours_from_anchor` column that makes the temporal position of each measurement
explicit.

This step defines the study window in an auditable, reproducible way — critical for
regulatory submissions and reproducibility.

In [ ]:
from clinops.temporal import CohortAligner

aligner = CohortAligner(
    anchor_col="intime",
    id_col="subject_id",
    time_col="charttime",
    max_hours_before=0,   # no pre-admission measurements
    max_hours_after=48,   # 48-hour observation window
)
aligned = aligner.align(events_df=charts, anchor_df=icustays)

print(f"Before alignment: {len(charts):,} measurements ({N_HOURS}h per patient)")
print(f"After  alignment: {len(aligned):,} measurements (48h per patient)")
print(f"hours_from_anchor range: [{aligned['hours_from_anchor'].min():.1f}, "
      f"{aligned['hours_from_anchor'].max():.1f}]")

## Step 5 — Temporal feature windows

We extract 8-hour windows with a 4-hour stride. Each window becomes one row in the
feature matrix, containing the mean vital signs for that patient during that period.

The 48-hour observation window with 4-hour stride produces up to 11 windows per patient,
giving the model the ability to see how vitals evolve over the ICU stay rather than
just a single snapshot.

In [ ]:
from clinops.temporal import TemporalWindower, ImputationStrategy

windower = TemporalWindower(
    window_hours=8,
    step_hours=4,
    imputation=ImputationStrategy.FORWARD_FILL,
    min_observations=2,
)
windows = windower.fit_transform(
    df=aligned,
    id_col="subject_id",
    time_col="charttime",
    feature_cols=["heart_rate", "spo2", "resp_rate", "map"],
)

print(f"Windows: {len(windows):,}  |  shape: {windows.shape}")
print(f"Windows per patient (avg): {len(windows) / N_PATIENTS:.1f}")
windows.head()

## Step 6 — Imputation (no leakage)

Clinical measurements are often missing — especially in noisier ICU data. We split the
windows into train and test first, then fit the imputer **only on the training set**.

This prevents test-set statistics from influencing how missing values are filled — a
subtle but real source of optimistic evaluation in clinical ML.

In [ ]:
from clinops.split import StratifiedPatientSplitter
from clinops.temporal import Imputer

# Attach outcome to windows for stratified splitting
outcome = admissions[["subject_id", "hospital_expire_flag"]]
windows_labeled = windows.merge(outcome, on="subject_id", how="left")

# Patient-stratified split (preserves mortality rate in both sets)
split = StratifiedPatientSplitter(
    id_col="subject_id",
    outcome_col="hospital_expire_flag",
    test_size=0.25,
    random_state=0,
).split(windows_labeled)

train_w = split.train.copy()
test_w  = split.test.copy()

print(split.summary())

# Inject some missingness into test vitals
rng2 = np.random.default_rng(7)
feat_cols = ["heart_rate", "spo2", "resp_rate", "map"]
for col in feat_cols:
    idx = rng2.choice(test_w.index, size=max(1, len(test_w) // 6), replace=False)
    test_w.loc[idx, col] = np.nan

missing_before = test_w[feat_cols].isna().sum().sum()
print(f"\nMissing values in test before imputation: {missing_before}")

# Fit on train only, transform test
imputer = Imputer(ImputationStrategy.MEAN)
imputer.fit(train_w[feat_cols])
test_w[feat_cols] = imputer.transform(test_w[feat_cols])[feat_cols]

print(f"Missing values in test after  imputation: {test_w[feat_cols].isna().sum().sum()}")

## Step 7 — Lag and rolling features

Lag features capture temporal trends that a single-window snapshot misses:
- A rising heart rate across three consecutive windows is more informative than
  the current heart rate alone.
- Rolling standard deviation captures instability (high variability in SpO2 is
  clinically concerning even if the mean is normal).

We apply this **after** splitting, on the train and test sets independently, using
the same `LagFeatureBuilder` configuration.

In [ ]:
from clinops.temporal import LagFeatureBuilder

builder = LagFeatureBuilder(
    lags=[1, 2],         # previous window and the one before that
    rolling_windows=[3], # 3-window rolling mean and std
    id_col="subject_id",
)

train_enriched = builder.fit_transform(train_w)
test_enriched  = builder.fit_transform(test_w)

lag_cols = [c for c in train_enriched.columns if c not in train_w.columns]
print(f"Added {len(lag_cols)} lag/rolling features:")
for c in lag_cols:
    print(f"  {c}")

## Step 8 — Merge comorbidities and build final feature matrix

We join the ICD-based comorbidity flags onto each window. This gives the model
access to both time-varying vital signs and static admission-level features in
a single flat table.

In [ ]:
def build_feature_matrix(df, comorbidities_df):
    """Merge comorbidity flags onto a windowed DataFrame."""
    return df.merge(comorbidities_df, on="subject_id", how="left")

train_final = build_feature_matrix(train_enriched, comorbidities)
test_final  = build_feature_matrix(test_enriched,  comorbidities)

# Identify feature columns (everything except IDs, timestamps, and the outcome)
non_feature = {"subject_id", "hadm_id", "stay_id", "window_start", "window_end",
               "charttime", "hospital_expire_flag"}
feature_cols = [c for c in train_final.columns if c not in non_feature]

X_train = train_final[feature_cols].values
y_train = train_final["hospital_expire_flag"].values

X_test  = test_final[feature_cols].values
y_test  = test_final["hospital_expire_flag"].values

print("=" * 50)
print("Final feature matrix")
print("=" * 50)
print(f"X_train : {X_train.shape}")
print(f"y_train : {y_train.shape}  (mortality rate: {y_train.mean():.2f})")
print(f"X_test  : {X_test.shape}")
print(f"y_test  : {y_test.shape}  (mortality rate: {y_test.mean():.2f})")
print(f"\nFeatures ({len(feature_cols)}):")
for c in feature_cols:
    print(f"  {c}")

## Step 9 — (Optional) Quick sanity check with a baseline model

This is outside the scope of `clinops` itself, but here is how the ML-ready arrays
slot into a scikit-learn workflow. Replace the `GradientBoostingClassifier` with
any model of your choice.

In [ ]:
try:
    from sklearn.ensemble import GradientBoostingClassifier
    from sklearn.metrics import roc_auc_score
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer

    clf = Pipeline([
        ("impute", SimpleImputer(strategy="mean")),   # handle any residual NaN from lag features
        ("model",  GradientBoostingClassifier(n_estimators=50, random_state=0)),
    ])
    clf.fit(X_train, y_train)

    proba = clf.predict_proba(X_test)[:, 1]
    auc   = roc_auc_score(y_test, proba)
    print(f"AUROC on test set (synthetic data): {auc:.3f}")
    print("(Performance on real MIMIC data will differ.)")
except ImportError:
    print("scikit-learn not installed — skipping baseline model.")
    print("Install with: pip install scikit-learn")

---

## Pipeline summary

| Stage | clinops component | Output |
|-------|-------------------|--------|
| Ingest & validate | `FlatFileLoader` + `ClinicalSchema` | Validated `DataFrame` |
| Clip outliers | `ClinicalOutlierClipper` | Physiologically-bounded vitals |
| ICD harmonization | `ICDMapper` | Unified ICD-10 codes + chapter groups |
| Comorbidity features | pandas pivot | Binary comorbidity matrix |
| Cohort alignment | `CohortAligner` | Events trimmed to 48h study window |
| Temporal windowing | `TemporalWindower` | One row per patient per 8h window |
| Split (no leakage) | `StratifiedPatientSplitter` | Patient-safe train / test sets |
| Imputation | `Imputer` | No NaN, no test-set leakage |
| Lag features | `LagFeatureBuilder` | Temporal trend features |
| Feature matrix | — | `X_train`, `y_train`, `X_test`, `y_test` |

## Adapting this notebook to real MIMIC-IV

Replace the synthetic data section with:

```python
from clinops.ingest import MimicTableLoader

tbl     = MimicTableLoader("/data/mimic-iv-2.2")
cohort  = [...]   # your list of subject_ids

charts  = tbl.chartevents(subject_ids=cohort)
icustays = tbl.icustays(subject_ids=cohort)
adm     = tbl.admissions(subject_ids=cohort)
dx      = tbl.diagnoses_icd(subject_ids=cohort)
```

Everything from Step 3 onwards is identical.

## Further reading

- [Getting started notebook](./01_getting_started.ipynb) — module-by-module walkthrough
- [Ingest guide](../../docs/guide/ingest.md) — all loader options including FHIR R4
- [Temporal guide](../../docs/guide/temporal.md) — windowing strategies, imputation tactics
- [Split guide](../../docs/guide/split.md) — choosing the right splitter for your study design